# D+ → π− π+ π+ toy-MC fit closure

End-to-end closure example for DalitzPlotFitter using the explicit final-state ordering

\[1=\pi^-,\qquad 2=\pi^+_1,\qquad 3=\pi^+_2.\]

Therefore, \(s_{12}=m^2(\pi^-\pi^+_1)\), \(s_{13}=m^2(\pi^-\pi^+_2)\), and \(s_{23}=m^2(\pi^+_1\pi^+_2)\). The two positive pions are identical and the amplitude is symmetrized accordingly.

1. build \(\rho(770)^0+f_0(980)+NR\);
2. inject known truth coefficients;
3. generate an independent toy sample;
4. randomize the starting values;
5. fit with cached line shapes and cached normalization matrix;
6. compare truth, start and fit;
7. plot the toy and the model before/after the fit.

Only magnitudes and phases float here, so the line shapes are evaluated once and reused throughout the minimization.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import mplhep as hep

from dalitzplotfitter import ThreeBodyPhaseSpace, enable_x64
from dalitzplotfitter.amplitude import (
    AmplitudeBuilder, AmplitudeComponent, PreparedAmplitudeCache,
    ConstantAmplitude, compile_amplitude_component,
    create_kinematic_transformer,
)
from dalitzplotfitter.coefficients import FitMagPhase
from dalitzplotfitter.fit import Minimizer, Parameter
from dalitzplotfitter.reaction import ReactionBuilder
from dalitzplotfitter.toy import ToyGenerator

enable_x64()
hep.style.use("LHCb2")

In [ ]:
def build_resonance(resonance):
    reaction = ReactionBuilder(
        initial_state="D+",
        final_state=["pi-", "pi+", "pi+"],
        allowed_intermediate_particles=[resonance],
    ).build()
    model = AmplitudeBuilder(reaction).build()
    return reaction, model, compile_amplitude_component(model)

rho_reaction, rho_model, rho_dynamics = build_resonance("rho(770)0")
_, _, f0_dynamics = build_resonance("f(0)(980)")

In [ ]:
truth = {"f0.r": 0.55, "f0.phi": 1.15, "nr.r": 0.28, "nr.phi": -0.85}

rho_r = Parameter.coefficient("rho.r", 1.0, fixed=True, owner="rho")
rho_phi = Parameter.coefficient("rho.phi", 0.0, fixed=True, owner="rho")

rng = np.random.default_rng(314159)
f0_r = Parameter.coefficient("f0.r", float(rng.uniform(0.25, 0.90)), bounds=(0, 1.5), step=0.02, owner="f0")
f0_phi = Parameter.coefficient("f0.phi", float(rng.uniform(-2.8, 2.8)), bounds=(-np.pi, np.pi), step=0.05, owner="f0")
nr_r = Parameter.coefficient("nr.r", float(rng.uniform(0.08, 0.65)), bounds=(0, 1.0), step=0.02, owner="NR")
nr_phi = Parameter.coefficient("nr.phi", float(rng.uniform(-2.8, 2.8)), bounds=(-np.pi, np.pi), step=0.05, owner="NR")

parameters = (rho_r, rho_phi, f0_r, f0_phi, nr_r, nr_phi)
initial = {p.name: p.value for p in parameters}

print("Truth:", truth)
print("Initial:", {k: initial[k] for k in truth})

In [ ]:
components = (
    AmplitudeComponent("rho", rho_dynamics, FitMagPhase(rho_r, rho_phi)),
    AmplitudeComponent("f0", f0_dynamics, FitMagPhase(f0_r, f0_phi)),
    AmplitudeComponent("NR", ConstantAmplitude(), FitMagPhase(nr_r, nr_phi)),
)
phase_space = ThreeBodyPhaseSpace.from_reaction(rho_reaction)
transformer = create_kinematic_transformer(rho_model)

def toy_intensity(data, values):
    rho = rho_dynamics(data, None)
    f0 = f0_dynamics(data, None)
    nr = ConstantAmplitude()(data, None)
    amp = (
        FitMagPhase(rho_r, rho_phi).value(values=values) * rho
        + FitMagPhase(f0_r, f0_phi).value(values=values) * f0
        + FitMagPhase(nr_r, nr_phi).value(values=values) * nr
    )
    return jnp.real(amp * jnp.conj(amp))

generator = ToyGenerator(phase_space, transformer, pool_size=60_000)
toy_sample, toy_data = generator.generate(
    jax.random.key(2026), size=3_000,
    intensity=toy_intensity, parameters=truth,
)
print(f"Generated {toy_sample.size} toy events")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
h = ax.hist2d(np.asarray(toy_sample.s12), np.asarray(toy_sample.s13), bins=70)
fig.colorbar(h[3], ax=ax, label="Candidates")
ax.set_xlabel(r"$m^2(\pi^-\pi^+_1)$ [GeV$^2$]")
ax.set_ylabel(r"$m^2(\pi^-\pi^+_2)$ [GeV$^2$]")
ax.set_title(r"Toy $D^+\to\pi^-\pi^+\pi^+$")
plt.show()

In [ ]:
normalization_sample = phase_space.generate(jax.random.key(2027), 60_000)
normalization_data = transformer(normalization_sample.as_momentum_dict())

cache = PreparedAmplitudeCache.prepare(
    components,
    data=toy_data,
    normalization_data=normalization_data,
    normalization_weights=normalization_sample.weights,
    parameters=parameters,
)

def nll(values):
    intensity, normalization = cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + toy_sample.size * jnp.log(normalization)

print("NLL truth  :", float(nll(truth)))
print("NLL initial:", float(nll(initial)))

In [ ]:
def model_weights(values):
    c = cache.coefficient_vector(values)
    amp = cache.normalization_components @ c
    intensity = jnp.real(amp * jnp.conj(amp))
    return np.asarray(normalization_sample.weights * intensity)

weights_initial = model_weights(initial)

def projection(ax, toy_x, mc_x, weights, bins=45, label="Model"):
    counts, edges = np.histogram(np.asarray(toy_x), bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    model, _ = np.histogram(np.asarray(mc_x), bins=edges, weights=weights)
    model *= counts.sum() / model.sum()
    ax.errorbar(centers, counts, yerr=np.sqrt(counts), fmt="o", label="Toy")
    ax.stairs(model, edges, label=label)
    return edges

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
vars_ = [
    (toy_sample.s12, normalization_sample.s12, r"$m^2(\pi^-\pi^+_1)$ [GeV$^2$]"),
    (toy_sample.s13, normalization_sample.s13, r"$m^2(\pi^-\pi^+_2)$ [GeV$^2$]"),
    (toy_sample.s23, normalization_sample.s23, r"$m^2(\pi^+_1\pi^+_2)$ [GeV$^2$]"),
]
for ax, (toy_x, mc_x, xlabel) in zip(axes, vars_):
    projection(ax, toy_x, mc_x, weights_initial, label="Before fit")
    ax.set_xlabel(xlabel); ax.set_ylabel("Candidates"); ax.legend()
fig.suptitle("Randomized starting model")
fig.tight_layout()
plt.show()

In [ ]:
result = Minimizer(nll, parameters).fit()
fit_values = {name: float(result.values[name]) for name in truth}
fit_errors = {name: float(result.errors[name]) for name in truth}

print(result)
for name in truth:
    print(f"{name:8s}: truth={truth[name]:+.4f}  start={initial[name]:+.4f}  fit={fit_values[name]:+.4f} ± {fit_errors[name]:.4f}")
print("NLL fit:", float(nll(fit_values)))

In [ ]:
def wrapped_delta(phi_fit, phi_true):
    return float(np.angle(np.exp(1j * (phi_fit - phi_true))))

print("Closure")
print("Δf0.r  =", fit_values["f0.r"] - truth["f0.r"])
print("Δf0.phi=", wrapped_delta(fit_values["f0.phi"], truth["f0.phi"]))
print("Δnr.r  =", fit_values["nr.r"] - truth["nr.r"])
print("Δnr.phi=", wrapped_delta(fit_values["nr.phi"], truth["nr.phi"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(2)

mag = ["f0.r", "nr.r"]
axes[0].scatter(x - .18, [truth[n] for n in mag], marker="*", s=160, label="Truth")
axes[0].scatter(x, [initial[n] for n in mag], marker="x", s=90, label="Initial")
axes[0].errorbar(x + .18, [fit_values[n] for n in mag], yerr=[fit_errors[n] for n in mag], fmt="o", label="Fit")
axes[0].set_xticks(x, [r"$f_0$", "NR"]); axes[0].set_ylabel("Magnitude"); axes[0].set_title("Magnitude closure"); axes[0].legend()

phase = ["f0.phi", "nr.phi"]
axes[1].scatter(x - .18, [truth[n] for n in phase], marker="*", s=160, label="Truth")
axes[1].scatter(x, [initial[n] for n in phase], marker="x", s=90, label="Initial")
axes[1].errorbar(x + .18, [fit_values[n] for n in phase], yerr=[fit_errors[n] for n in phase], fmt="o", label="Fit")
axes[1].set_xticks(x, [r"$f_0$", "NR"]); axes[1].set_ylabel("Phase [rad]"); axes[1].set_title("Phase closure"); axes[1].legend()
fig.tight_layout(); plt.show()

In [ ]:
weights_fit = model_weights(fit_values)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (toy_x, mc_x, xlabel) in zip(axes, vars_):
    counts, edges = np.histogram(np.asarray(toy_x), bins=45)
    centers = 0.5 * (edges[:-1] + edges[1:])
    h0, _ = np.histogram(np.asarray(mc_x), bins=edges, weights=weights_initial)
    h1, _ = np.histogram(np.asarray(mc_x), bins=edges, weights=weights_fit)
    h0 *= counts.sum()/h0.sum(); h1 *= counts.sum()/h1.sum()
    ax.errorbar(centers, counts, yerr=np.sqrt(counts), fmt="o", label="Toy")
    ax.stairs(h0, edges, label="Before fit")
    ax.stairs(h1, edges, label="After fit")
    ax.set_xlabel(xlabel); ax.set_ylabel("Candidates"); ax.legend()
fig.suptitle("Toy projections: before and after the fit")
fig.tight_layout(); plt.show()

In [ ]:
bins = 65
xedges = np.linspace(
    min(float(jnp.min(toy_sample.s12)), float(jnp.min(normalization_sample.s12))),
    max(float(jnp.max(toy_sample.s12)), float(jnp.max(normalization_sample.s12))),
    bins + 1,
)
yedges = np.linspace(
    min(float(jnp.min(toy_sample.s13)), float(jnp.min(normalization_sample.s13))),
    max(float(jnp.max(toy_sample.s13)), float(jnp.max(normalization_sample.s13))),
    bins + 1,
)
toy_h, _, _ = np.histogram2d(np.asarray(toy_sample.s12), np.asarray(toy_sample.s13), bins=[xedges, yedges])
start_h, _, _ = np.histogram2d(np.asarray(normalization_sample.s12), np.asarray(normalization_sample.s13), bins=[xedges, yedges], weights=weights_initial)
fit_h, _, _ = np.histogram2d(np.asarray(normalization_sample.s12), np.asarray(normalization_sample.s13), bins=[xedges, yedges], weights=weights_fit)

toy_h /= toy_h.sum(); start_h /= start_h.sum(); fit_h /= fit_h.sum()
vmax = max(toy_h.max(), start_h.max(), fit_h.max())

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, hist, title in zip(axes, [toy_h, start_h, fit_h], ["Toy data", "Model before fit", "Model after fit"]):
    mesh = ax.pcolormesh(xedges, yedges, hist.T, shading="auto", vmin=0, vmax=vmax)
    ax.set_xlabel(r"$m^2(\pi^-\pi^+_1)$ [GeV$^2$]")
    ax.set_ylabel(r"$m^2(\pi^-\pi^+_2)$ [GeV$^2$]")
    ax.set_title(title)
fig.colorbar(mesh, ax=axes, label="Normalized bin content")
plt.show()

A successful closure should show a valid minimum, fitted values compatible with the injected truth, lower NLL after minimization, and visibly improved agreement between the toy projections and the fitted model.

Because this is a coefficient-only fit, the cached \(F_i(x)\) values and \(M_{ij}\) matrix are reused throughout the minimization.